# 🧠 Hybrid Classification Model with RoBERTa + Tabular Data
This tutorial shows how to combine `review_text` with tabular features like `rating`, `delivery_delay`, and `is_verified` to build a binary classifier using RoBERTa + feedforward layers.

In [1]:
# 📦 Install necessary libraries
!pip install transformers datasets scikit-learn

In [2]:
# ✅ Imports
import torch
from transformers import RobertaTokenizer, RobertaModel
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch.nn as nn
import torch.optim as optim

In [3]:
# 🧪 Simulated dataset mimicking Oracle table
texts = [
    "Great product, works as expected.",
    "Looks nice, haven't tried it yet.",
    "Very satisfied with the performance.",
    "Package arrived on time, not used yet.",
    "Loved it! Highly recommended.",
    "Nice packaging, still evaluating.",
    "Absolutely wonderful. Will buy again.",
    "Seems fine, not tested yet.",
    "Excellent quality and easy to use.",
    "Got it yesterday, looks okay so far."
]

labels = [
    "positive", "positive_irrelevant", "positive", "positive_irrelevant",
    "positive", "positive_irrelevant", "positive", "positive_irrelevant",
    "positive", "positive_irrelevant"
]

df = pd.DataFrame({
    'review_text': texts,
    'rating': np.random.randint(3, 6, size=10),
    'delivery_delay': np.random.randint(0, 5, size=10),
    'is_verified': np.random.choice([0, 1], size=10),
    'label': labels
})
df.head()

,review_text,rating,delivery_delay,is_verified,label
0,"Great product, works as expected.",3,2,0,positive
1,"Looks nice, haven't tried it yet.",4,3,0,positive_irrelevant
2,Very satisfied with the performance.,4,4,1,positive
3,"Package arrived on time, not used yet.",3,3,0,positive_irrelevant
4,Loved it! Highly recommended.,4,2,1,positive


In [4]:
# 🔤 Tokenize review_text
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokens = tokenizer(list(df['review_text']), padding=True, truncation=True, return_tensors='pt')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [5]:
# 🔎 RoBERTa embeddings (CLS token)
model = RobertaModel.from_pretrained("roberta-base")
with torch.no_grad():
    outputs = model(**tokens)
    cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# 🧮 Normalize tabular features
scaler = StandardScaler()
tabular = scaler.fit_transform(df[['rating', 'delivery_delay', 'is_verified']])

In [7]:
# 🔀 Combine text + tabular
X = np.concatenate([cls_embeddings, tabular], axis=1)
y = LabelEncoder().fit_transform(df['label'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
# 🤖 Simple classifier
class Classifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.net(x)

model = Classifier(X.shape[1])
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [9]:
# 🏋️ Train loop
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

for epoch in range(15):
    opt.zero_grad()
    pred = model(X_train_tensor)
    loss = loss_fn(pred, y_train_tensor)
    loss.backward()
    opt.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

Epoch 1: Loss = 0.6920
Epoch 2: Loss = 0.6770
Epoch 3: Loss = 0.6663
Epoch 4: Loss = 0.6580
Epoch 5: Loss = 0.6492
Epoch 6: Loss = 0.6394
Epoch 7: Loss = 0.6294
Epoch 8: Loss = 0.6191
Epoch 9: Loss = 0.6082
Epoch 10: Loss = 0.5970
Epoch 11: Loss = 0.5855
Epoch 12: Loss = 0.5737
Epoch 13: Loss = 0.5617
Epoch 14: Loss = 0.5489
Epoch 15: Loss = 0.5355


In [10]:
# 📊 Evaluate
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    pred_logits = model(X_test_tensor)
    y_pred = torch.argmax(pred_logits, dim=1).numpy()

print(classification_report(y_test, y_pred, target_names=['positive', 'positive_irrelevant']))

                     precision    recall  f1-score   support

           positive       0.33      1.00      0.50         1
positive_irrelevant       0.00      0.00      0.00         2

           accuracy                           0.33         3
          macro avg       0.17      0.50      0.25         3
       weighted avg       0.11      0.33      0.17         3



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
